# Indicadores de seguimiento de la evaluación del PNIE <br> Parte VII: Estimación de cierre de brechas

Autor: Equipo de Evaluación del PNIE

Fecha de creación: 16/10/2025

Fecha de actualización: 27/10/2025

## 01. Configuración del notebook

In [11]:
# pip install xlsxwriter

In [12]:
# Importando librerias
import os
import numpy as np
import pandas as pd
# from dbfread import DBF
import re
import unicodedata
from typing import Optional
import matplotlib.pyplot as plt

In [13]:
# Directorio de trabajo
file1 = 'E:/OneDrive - Ministerio de Educación/00. Paolo/Actividades'
file1 = 'D:/OneDrive - Ministerio de Educación/00. Paolo/Actividades'
file2 = '01. Evaluación PNIE/02. IS-PNIE'
file = file1 + '/' + file2
os.chdir(file)

In [14]:
# Configurando el formato
pd.options.mode.chained_assignment = None
pd.options.display.float_format = '{:,.1f}'.format

In [15]:
# Definiendo funciones de conteo
def unique(data, varlist):
    aux = data[varlist]
    a = aux.drop_duplicates().shape[0]
    list = " ".join(varlist)
    print(f'Number of unique values of {list} is {a:,.0f}')
    print(f'Number of records is {data.shape[0]:,.0f}')

def by_unique(data, varlist, var):
    unique(data, varlist)
    list = varlist + var
    aux = data[list]
    a = aux.drop_duplicates().value_counts(var).reset_index().\
    sort_values(by=var)
    print("\n", a)

In [16]:
# Corrección de codigos locales
def codlocal(df, oldvar, newvar):
    df[newvar] = df[oldvar].astype(str).str.zfill(6)

In [17]:
def freq(df, columna, dropna=False, decimales=1):
    """
    Genera una tabla de frecuencias absolutas, relativas (%) y totales.

    Parámetros:
    -----------
    df : DataFrame
        El DataFrame de entrada
    columna : str
        Nombre de la columna a analizar
    dropna : bool, opcional
        Si True, ignora los valores NaN. Default = False
    decimales : int, opcional
        Número de decimales para el porcentaje. Default = 2
    """
    
    # Frecuencias absolutas y relativas
    freq = df[columna].value_counts(dropna=dropna)
    rel = df[columna].value_counts(normalize=True, dropna=dropna) * 100
    
    # Construcción de la tabla
    tabla = pd.DataFrame({
        'Freq': freq,
        '(%)': rel.round(decimales)
    })
    
    # Totales
    total_freq = tabla['Freq'].sum()
    tabla.loc['TOTAL'] = [total_freq, 100]

    # Formateo: Freq con separador de miles y (%) con % explícito
    tabla['Freq'] = tabla['Freq'].apply(lambda x: f"{int(x):,}" if pd.notna(x) else x)
    tabla['(%)'] = tabla['(%)'].apply(lambda x: f"{x:.{decimales}f}%" if x != 100 else "100%")
 
    return print(tabla, '\n')

## 02. Estimación del cierre de brechas de los GI y OE

### 2.1 PNIE

In [ ]:
df = pd.read_csv("data/procesadas/indicadores_PNIE.csv", dtype={'codlocal': 
                                                                str})
bd = pd.read_stata('data/LE_BasePr_PNIE.dta')
codlocal(bd, 'id_local', 'codlocal')

df = pd.merge(df, bd, on='codlocal', how='left', indicator='_PNIE')
freq(df, '_PNIE')
df.drop(columns=['_PNIE'], inplace=True)

df = df[['codlocal', 'GI1', 'GI2', 'GI3_1', 'GI3_2', 'GI3_3', 'GI3_4', 
         'GI3', 'GI4', 'GI5', 'OE1', 'OE4',
         'CT_grupo1', 'CT_grupo2', 'CT_grupo3', 'CT_grupo4', 'CT_grupo5',
         'COSTO_TOTAL', 'costo_prom', 'area_sig', 'areatechada', 'area_dem',
         'area_ri', 'area_rc', 'CTot_repLOC', 'costoSAFIL',
         'CT_CalidadEle', 'Mobiliario_E', 'CTot_ENE']]

# Estimación de la brecha de mantenimiento preventivo
df['cu'] = df['costo_prom'].min()
df['fcr'] = np.where(df['area_sig'] == 'Urbana', 1, 0.94)
df['area'] = df['areatechada'] - (df['area_dem'] + df['area_ri'] + \
                                  df['area_rc']) + 0.0001
df.loc[df['area'] < 0, 'area'] = 0
df['val_act'] = df['area'] * df['cu'] * df['fcr'] * 1.05 * 1.20 * 1.18
df['brecha_mp'] = df['val_act'] * 0.01
mask = ((df['COSTO_TOTAL'].isna()) | (df['CTot_repLOC'].gt(0)))
df.loc[mask, 'brecha_mp'] = np.nan
df.drop(columns=['cu', 'fcr', 'area', 'val_act', 'costo_prom', 'area_sig', 
                 'areatechada', 'area_dem', 'area_ri', 'area_rc', 
                 'CTot_repLOC'], inplace=True)

# Estimación de la brecha asociada al GI3
df['GI3_brecha'] = np.sum(df[['CT_grupo3', 'brecha_mp']], axis=1)

df['GI1_brecha_post'] = np.where(df['GI1'] == 1, 0, df['CT_grupo1'])
df['GI2_brecha_post'] = np.where(df['GI2'] == 1, 0, df['CT_grupo2'])
df['GI3_brecha_post'] = np.where(df['GI3'] == 1, 0, df['GI3_brecha'])
df['GI4_brecha_post'] = np.where(df['GI4'] == 1, 0, df['CT_grupo4'])
df['GI5_brecha_post'] = np.where(df['GI5'] == 1, 0, df['CT_grupo5'])

# Exportando resultados a nivel de codlocal
df.to_csv("data/procesadas/cierre_brecha_PNIE.csv", index=False)

# Estimando brecha post
df = df[['CT_grupo1', 'GI1_brecha_post', 'CT_grupo2', 'GI2_brecha_post',
         'GI3_brecha','GI3_brecha_post', 'CT_grupo4', 'GI4_brecha_post',
         'CT_grupo5', 'GI5_brecha_post', 'costoSAFIL', 'COSTO_TOTAL',
         'CT_CalidadEle', 'Mobiliario_E', 'CTot_ENE',
         'GI3_1', 'GI3_2', 'GI3_3', 'GI3_4',]]

# Estimación de la brecha OE1 y OE4
df['OE1_pre'] = np.sum(df[['CT_grupo1', 'CT_grupo2', 'CT_CalidadEle', 
                           'Mobiliario_E', 'CT_grupo4', 'CT_grupo5', 
                           'costoSAFIL']], axis=1)

df['ct_ce_post'] = np.where(df['GI3_1'] == 1, 0, df['CT_CalidadEle'])
df['ct_me_post'] = np.where(df['GI3_2'] == 1, 0, df['Mobiliario_E'])



# df['OE1_post'] = 



# df['brecha_post'] = np.sum(df[['GI1_brecha_post', 'GI2_brecha_post',
#                                'GI3_brecha_post', 'GI4_brecha_post',
#                                'GI5_brecha_post', 'costoSAFIL']], axis=1)
df.columns

C:\Users\Paolo\AppData\Local\Temp\ipykernel_18620\3871407433.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[newvar] = df[oldvar].astype(str).str.zfill(6)


              Freq   (%)
_PNIE                   
both        42,331  100%
left_only        0  0.0%
right_only       0  0.0%
TOTAL       42,331  100% 



Index(['CT_grupo1', 'GI1_brecha_post', 'CT_grupo2', 'GI2_brecha_post',
       'GI3_brecha', 'GI3_brecha_post', 'CT_grupo4', 'GI4_brecha_post',
       'CT_grupo5', 'GI5_brecha_post', 'costoSAFIL', 'COSTO_TOTAL',
       'CT_CalidadEle', 'Mobiliario_E', 'CTot_ENE'],
      dtype='object')

### 2.2 2019-2025

In [18]:
def av_cb(anio, summary=False):
    df = pd.read_csv(f"data/procesadas/indicadores_{anio}.csv", 
                     dtype={'codlocal': str})
    bd = pd.read_stata(f"data/LE_BasePr_{anio}.dta")
    codlocal(bd, 'cod_local', 'codlocal')

    df = pd.merge(df, bd, on='codlocal', how='left', indicator=True)
#     freq(df, '_merge')
    df.drop(columns=['_merge'], inplace=True)

    # Estimación de la brecha de mantenimiento preventivo
    df['cu'] = df['cu_atyoe'].min()
    df['fcr'] = np.where(df['urbano'] == 1, 1, 0.94)
    df['area'] = df['areatech'] - (df['areadem'] + df['areari'] + \
                                   df['arearc']) + 0.0001
    df['val_act'] = df['area'] * df['cu'] * df['fcr'] * 1.05 * 1.20 * 1.18
    df['ct_mp'] = df['val_act'] * 0.01
    mask = ((df['finfo'] == 'Sin información') | 
            (df['prioridad'] == '1: Riesgo'))
    df.loc[mask, 'ct_mp'] = np.nan
    df.drop(columns=['cu', 'fcr', 'area', 'val_act'], inplace=True)

    # GI1 (falta brecha Plan Selva)
    df['ct_st_d_post'] = np.where(df['GI1_1'] == 1, 0, df['ct_st_d'])
    df['ct_sp_d_post'] = np.where(df['GI1_1'] == 1, 0, df['ct_sp_d'])
    df['ct_ri_post'] = np.where(df['GI1_2'] == 1, 0, df['ct_ri'])
    df['ct_ic_post'] = np.where(df['GI1_3'] == 1, 0, df['ct_ic'])
    df['ct_cp_post'] = np.where(df['GI1_5'] == 1, 0, df['ct_cp'])
    df['b_1_proxy_post'] = np.sum(df[['ct_st_d_post', 'ct_sp_d_post', 
                                      'ct_ri_post', 'ct_ic_post', 
                                      'ct_cp_post']], axis=1)
    df.loc[df['GI1'] == 1, 'b_1_proxy_post'] = 0

    # GI2
    df['ct_aad_post'] = np.where(df['GI2_1'] == 1, 0, df['ct_aad'])
    df['ct_cad_post'] = np.where(df['GI2_2'] == 1, 0, df['ct_cad'])
    df['b_2_post'] = np.sum(df[['ct_aad_post', 'ct_cad_post']], axis=1)
    df.loc[df['GI2'] == 1, 'b_2_post'] = 0

    # GI3 (se incluye la brecha de mantenimiento preventivo)
    df['ct_ce_post'] = np.where(df['GI3_1'] == 1, 0, df['ct_ce'])
    df['ct_st_me_post'] = np.where(df['GI3_2'] == 1, 0, df['ct_st_me'])
    df['ct_sp_me_post'] = np.where(df['GI3_2'] == 1, 0, df['ct_sp_me'])
    df['ct_ri_me_post'] = np.where(df['GI3_2'] == 1, 0, df['ct_ri_me'])
    df['ct_rc_me_post'] = np.where(df['GI3_2'] == 1, 0, df['ct_rc_me'])
    df['ct_ic_me_post'] = np.where(df['GI3_2'] == 1, 0, df['ct_ic_me'])
    df['ct_amp_me_post'] = np.where(df['GI3_2'] == 1, 0, df['ct_amp_me'])
    df['ct_ene_post'] = np.where(df['GI3_3'] == 1, 0, df['ct_ene'])
    df['ct_mp_post'] = np.where(df['GI3_4'] == 1, 0, df['ct_mp'])
    df['b_3_post'] = np.sum(df[['ct_ce_post', 'ct_st_me_post', 'ct_sp_me_post', 
                                'ct_ri_me_post', 'ct_rc_me_post', 
                                'ct_ic_me_post', 'ct_amp_me_post', 
                                'ct_ene_post', 'ct_mp_post']], axis=1)
    df.loc[df['GI3'] == 1, 'b_3_post'] = 0

    df['b_3_cabrpr_post'] = np.sum(df[['ct_ce_post', 'ct_st_me_post', 
                                    'ct_sp_me_post', 'ct_ri_me_post', 
                                    'ct_rc_me_post', 'ct_ic_me_post',
                                    'ct_amp_me_post', 'ct_ene_post']], axis=1)
    df.loc[df['GI3'] == 1, 'b_3_cabrpr_post'] = 0

    # GI4
    df['ct_sp_r_post'] = np.where(df['GI4_1'] == 1, 0, df['ct_sp_r'])
    df['ct_ae_post'] = np.where(df['GI4_2'] == 1, 0, df['ct_ae'])
    df['ct_acc_post'] = np.where(df['GI4_3'] == 1, 0, df['ct_acc'])
    df['ct_amp_post'] = np.where(df['GI4_4'] == 1, 0, df['ct_amp'])
    df['ct_rc_post'] = np.where(df['GI1_2'] == 1, 0, df['ct_rc'])
    df['ct_ic_r_post'] = np.where(df['GI1_3'] == 1, 0, df['ct_ic_r'])
    df['b_4_post'] = np.sum(df[['ct_sp_r_post', 'ct_rc_post', 'ct_ic_r_post', 
                                'ct_amp_post', 'ct_ae_post', 'ct_acc_post']], 
                            axis=1)
    df.loc[df['GI4'] == 1, 'b_4_post'] = 0

    # GI5 (falta brecha de nueva infraestructura)
    df['ct_st_ra_post'] = np.where(df['GI5_1'] == 1, 0, df['ct_st_ra'])
    df['ct_st_ae_post'] = np.where(df['GI5_1'] == 1, 0, df['ct_st_ae'])
    df['ct_st_aad_post'] = np.where(df['GI5_1'] == 1, 0, df['ct_st_aad'])
    df['b_5_proxy_post'] = np.sum(df[['ct_st_ra_post', 'ct_st_ae_post', 
                                    'ct_st_aad_post']], axis=1)
    df.loc[df['GI5_1'] == 1, 'b_5_proxy_post'] = 0

    # Estimación de la brecha asociada al OE1 (falta Plan Selva)
    var_oe1 = ['ct_st_d', 'ct_sp_d', 'ct_ri', 'ct_ic', 'ct_cp',
            'ct_aad', 'ct_cad',
            'ct_ce', 'ct_st_me', 'ct_sp_me', 'ct_ri_me', 'ct_rc_me', 'ct_ic_me', 
            'ct_amp_me',
            'ct_sp_r', 'ct_ae', 'ct_acc', 'ct_amp', 'ct_rc', 'ct_ic_r',
            'ct_st_ra', 'ct_st_ae', 'ct_st_aad',
            'b_safil_ini']
    df['brecha_OE1_proxy'] = np.sum(df[var_oe1], axis=1)
    var_oe1 = ['ct_st_d_post', 'ct_sp_d_post', 'ct_ri_post', 'ct_ic_post', 
            'ct_cp_post',
            'ct_aad_post', 'ct_cad_post',
            'ct_ce_post', 'ct_st_me_post', 'ct_sp_me_post', 'ct_ri_me_post', 
            'ct_rc_me_post', 'ct_ic_me_post', 'ct_amp_me_post',
            'ct_sp_r_post', 'ct_ae_post', 'ct_acc_post', 'ct_amp_post', 
            'ct_rc_post', 'ct_ic_r_post',
            'ct_st_ra_post', 'ct_st_ae_post', 'ct_st_aad_post',
            'b_safil']
    df['brecha_OE1_proxy_post'] = np.sum(df[var_oe1], axis=1)

    # Estimación de la brecha asociada al OE4
    df['brecha_OE4'] = np.sum(df[['ct_ene', 'ct_mp']], axis=1)
    df['brecha_OE4_post'] = np.sum(df[['ct_ene_post', 'ct_mp_post']], axis=1)

    # Exportando resultados a nivel de codlocal
    df.to_csv(f"data/procesadas/LE_cierre_brecha_{anio}.csv", index=False)

    # Estimando avance del cierre de brecha
    df = df[['b_1_ini', 'b_1', 'b_1_proxy_post',
            'b_2_ini', 'b_2', 'b_2_post',
            'b_3_ini', 'b_3', 'b_3_post', 'b_3_cabrpr_post',
            'b_4_ini', 'b_4', 'b_4_post',
            'b_5_ini', 'b_5', 'b_5_proxy_post',
            'b_safil_ini', 'b_safil',
            'brecha_ini', 'brecha',
            'brecha_OE1_proxy', 'brecha_OE1_proxy_post',
            'brecha_OE4', 'brecha_OE4_post']].sum(axis=0).reset_index()
    df.rename(columns={0: 'valor', 'index': 'grupo'}, inplace=True)

    df['ind_ini'] = df['grupo'].str.contains('ini').astype(int)
    df.loc[df['grupo'] == 'brecha_OE1_proxy', 'ind_ini'] = 1
    df.loc[df['grupo'] == 'brecha_OE4', 'ind_ini'] = 1
    df['ind_post'] = df['grupo'].str.contains('post').astype(int)
    df.loc[df['grupo'] == 'b_3_cabrpr_post', 'ind_post'] = 0
    df['_df'] = df['ind_ini'] + df['ind_post']
    df['ind_plani'] = np.where(df['_df'] == 0, 1, 0)
    df.loc[df['grupo'] == 'b_3_cabrpr_post', 'ind_plani'] = 0

    df['grupos'] = ''
    df.loc[df['grupo'].str.contains('b_1'), 'grupos'] = 'GI1'
    df.loc[df['grupo'].str.contains('b_2'), 'grupos'] = 'GI2'
    df.loc[df['grupo'].str.contains('b_3'), 'grupos'] = 'GI3'
    df.loc[df['grupo'].str.contains('b_4'), 'grupos'] = 'GI4'
    df.loc[df['grupo'].str.contains('b_5'), 'grupos'] = 'GI5'
    df.loc[df['grupo'].str.contains('b_safil'), 'grupos'] = 'SAFIL'
    df.loc[df['grupo'].str.contains('OE1_proxy'), 'grupos'] = 'OE1'
    df.loc[df['grupo'].str.contains('OE4'), 'grupos'] = 'OE4'
    df.loc[df['grupo'] == 'brecha_ini', 'grupos'] = 'brecha_total'
    df.loc[df['grupo'] == 'brecha', 'grupos'] = 'brecha_total'

    part1 = df.loc[df['ind_ini'] == 1, ['grupos', 'valor']]
    part2 = df.loc[df['ind_post'] == 1, ['grupos', 'valor']]
    part3 = df.loc[df['ind_plani'] == 1, ['grupos', 'valor']]

    df = pd.merge(part1, part3, on='grupos', how='left', suffixes=('_ini', 
                                                                '_plani'))
    df = pd.merge(df, part2, on='grupos', how='left', suffixes=('', '_post'))
    df.loc[df['grupos'] == 'SAFIL', 'valor'] = df['valor_plani']

    df['avance_plani'] = abs((df['valor_plani'] / df['valor_ini'] - 1) * 100)
    df['avance_pnie'] = abs((df['valor'] / df['valor_ini'] - 1) * 100)

    # Exportando resultados a nivel de grupos
    df.to_csv(f"data/procesadas/grupos_cierre_brecha_{anio}.csv", index=False)

    # Exportando resultados finales
    df = df[['grupos', 'avance_plani', 'avance_pnie']]
    df = df.loc[df['grupos'] != 'brecha_total']
    df.to_csv(f"data/procesadas/cierre_brecha_{anio}.csv", index=False)

    if summary:
        print(f'\nAvance cierre brecha: {anio}')
        print(df, '\n')

In [21]:
anios = ['2019', '2020', '2021', '2022', '2023', '2024', '2025']

for anio in anios:
    av_cb(anio)

avances = pd.DataFrame()
for anio in anios:
    df = pd.read_csv(f"data/procesadas/cierre_brecha_{anio}.csv")
    df.rename(columns={'avance_plani': f'avance_plani_{anio}',
                       'avance_pnie': f'avance_pnie_{anio}'}, inplace=True)
    avances = pd.merge(avances, df, on='grupos', how='outer') if not \
        avances.empty else df
    
avances = avances.T.reset_index()
avances.rename(columns=avances.iloc[0], inplace=True)
avances = avances[1:]
avances['equipo'] = np.where(avances['grupos'].str.contains('plani'), 
                           'DIPLAN', 'Equ. Eval. PNIE')
avances['anio'] = avances['grupos'].str.extract(r'(\d{4})').astype(str)
avances = avances[['anio', 'equipo', 'GI1', 'GI2', 'GI3', 'GI4', 'GI5',
                   'SAFIL', 'OE1', 'OE4']]

part1 = avances[avances['equipo'] == 'Equ. Eval. PNIE']
part1.drop(columns=['equipo'], inplace=True)
part2 = avances[avances['equipo'] == 'DIPLAN']
part2.drop(columns=['equipo', 'OE1', 'OE4'], inplace=True)
part2.rename(columns={'GI1': 'GI1_DIPLAN', 'GI2': 'GI2_DIPLAN', 
                      'GI3': 'GI3_DIPLAN', 'GI4': 'GI4_DIPLAN', 
                      'GI5': 'GI5_DIPLAN', 'SAFIL': 'SAFIL_DIPLAN'}, 
                      inplace=True)
avances = pd.merge(part1, part2, on='anio', how='left')

avances.to_excel("data/procesadas/avances_cierre_brecha.xlsx", index=False)

C:\Users\diplan27.MINEDU\AppData\Local\Temp\ipykernel_4892\3871407433.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[newvar] = df[oldvar].astype(str).str.zfill(6)
C:\Users\diplan27.MINEDU\AppData\Local\Temp\ipykernel_4892\3871407433.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[newvar] = df[oldvar].astype(str).str.zfill(6)
C:\Users\diplan27.MINEDU\AppData\Local\Temp\ipykernel_4892\1176742972.py:4: UnicodeWarning: 
One or more strings in the dta file could not be decoded using utf-8, and
so the fallback encoding 

In [36]:
anios = ['2019', '2020', '2021', '2022', '2023', '2024', '2025']

oes_total = pd.DataFrame()
for anio in anios:
    aux = pd.read_csv(f"data/procesadas/grupos_cierre_brecha_{anio}.csv")
    oes = ['OE1', 'OE4']
    aux = aux.loc[aux['grupos'].isin(oes), :]
    aux['anio'] = anio
    oes_total = pd.concat([oes_total, aux], ignore_index=True)

df_wide = oes_total.pivot(
    index="anio",          # las filas serán los años
    columns="grupos",      # las columnas serán OE1, OE4, ...
    values=["valor_ini", "valor", "avance_pnie"]  # qué variables pivotear
)
df_wide.columns = [f"{var}_{grp}" for var, grp in df_wide.columns]
df_wide = df_wide.reset_index()

df_wide = df_wide[['anio', 
                   'valor_ini_OE1', 'valor_OE1', 'avance_pnie_OE1', 
                   'valor_ini_OE4', 'valor_OE4', 'avance_pnie_OE4']]

df_wide.to_excel("data/procesadas/OE_cierre_brecha.xlsx", index=False)

9.0

## 03. Cuadro de indicadores: numerador, denominador y valor

In [225]:
anios = ['PNIE', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
ind = pd.DataFrame()

for anio in anios:
    df = pd.read_csv(f"data/procesadas/indicadores_{anio}.csv", 
                     dtype={'codlocal': str})
    
    sum_cols  = ['GI5_2', 'GI5', 'OE2', 'RF2_matri', 'matricula']
    mean_cols = ['GI1_1', 'GI1_2', 'GI1_3', 'GI1_4', 'GI1_5', 
                 'GI2_1', 'GI2_2', 
                 'GI3_1', 'GI3_2', 'GI3_3', 'GI3_4', 
                 'GI4_1', 'GI4_2', 'GI4_3', 'GI4_4', 
                 'GI5_1',
                 'GI1', 'GI2', 'GI3', 'GI4', 'ET1', 'ET2', 'OE1', 'OE4', 'RF1']

    agg = {"codlocal": pd.Series.nunique}
    for c in df.columns:
        if c in sum_cols:
            agg[c] = "sum"
        elif c in mean_cols:
            agg[c] = "mean"

    out = df.agg(agg).to_frame().T
    out.insert(0, "anio", anio)
    out = out[["anio", "codlocal"] + [c for c in out.columns if c 
                                      not in ["anio", "codlocal"]]]
    
    # Multiplicar los promedios por 100 y redondear a 1 decimal
    for c in mean_cols:
        if c in out.columns:
            out[c] = (out[c] * 100).round(1)

    # Redondear las sumas a enteros
    sum_cols  = ['codlocal', 'GI5_2', 'GI5', 'OE2', 'RF2_matri', 'matricula']
    for c in sum_cols:
        if c in out.columns:
            out[c] = out[c].round(0).astype("Int64") 

    # Concatenar al dataframe final
    ind = pd.concat([ind, out], axis=0)

ind.reset_index(drop=True, inplace=True)
ind['RF2'] = ind['RF2_matri'] / ind['matricula'] * 100
ind.drop(columns=['RF2_matri', 'matricula'], inplace=True)
ind
df_ind = ind.T.copy()
df_ind.columns = df_ind.iloc[0]
df_ind = df_ind[1:]
df_ind = df_ind.reset_index().rename(columns={'index': 'Grupos de Intervención'})
df_ind.reset_index(drop=True, inplace=True)
df_ind

anio,Grupos de Intervención,PNIE,2019,2020,2021,2022,2023,2024,2025
0,codlocal,42331,54973,55058,55211,55304,55358,55425,55602
1,GI1_1,1.8,8.8,6.8,8.8,6.2,4.7,3.9,4.3
2,GI1_2,0.5,8.3,8.3,12.1,10.2,9.7,9.2,8.7
3,GI1_3,0.4,8.8,7.7,9.0,9.0,8.7,7.5,6.4
4,GI1_4,0.0,5.9,4.7,5.6,3.6,2.9,2.4,2.3
5,GI1_5,0.9,14.3,11.5,15.8,17.4,15.9,20.4,18.4
6,GI2_1,30.7,50.9,55.2,38.6,41.6,39.1,44.0,43.8
7,GI2_2,0.3,10.2,8.6,10.9,15.1,15.8,15.0,14.1
8,GI3_1,0.3,9.4,7.8,9.8,13.8,14.3,13.8,12.7
9,GI3_2,0.8,9.6,7.4,9.2,8.4,14.7,10.1,14.8


In [53]:
anios = ['PNIE', '2019', '2020', '2021', '2022', '2023', '2024', '2025']

ind = pd.DataFrame()

for anio in anios:
    df = pd.read_csv(f"data/procesadas/indicadores_{anio}.csv",
                    dtype={'codlocal': str})

    sum_cols  = ['GI1_1', 'GI1_2', 'GI1_3', 'GI1_4', 'GI1_5', 
                'GI2_1', 'GI2_2', 
                'GI3_1', 'GI3_2', 'GI3_3', 'GI3_4', 
                'GI4_1', 'GI4_2', 'GI4_3', 'GI4_4', 
                'GI5_1', 'GI5_2',
                'GI1', 'GI2', 'GI3', 'GI4', 'GI5',
                'ET1', 'ET2', 'OE1', 'OE2', 'OE4', 
                'RF1', 'RF2_matri', 'matricula']
    count_cols = ['GI1_1', 'GI1_2', 'GI1_3', 'GI1_4', 'GI1_5', 
                'GI2_1', 'GI2_2', 
                'GI3_1', 'GI3_2', 'GI3_3', 'GI3_4', 
                'GI4_1', 'GI4_2', 'GI4_3', 'GI4_4', 
                'GI5_1',
                'GI1', 'GI2', 'GI3', 'GI4', 'ET1', 'ET2', 'OE1', 'OE4', 'RF1']

    # Dimensionando a nivel de indicador
    res = {'codlocal': df['codlocal'].nunique()}
    sums = df[sum_cols].sum()
    res.update({f'{c}_num': float(sums[c]) for c in sum_cols})
    counts = df[count_cols].count()
    res.update({f'{c}_den': int(counts[c]) for c in count_cols})
    out_cols = (['codlocal']
                + [f'{c}_num'   for c in sum_cols]
                + [f'{c}_den' for c in count_cols])
    out = pd.DataFrame([res])[out_cols]
    out.rename(columns={'codlocal': 'codlocal_num',
                        'RF2_matri_num': 'RF2_num',
                        'matricula_num': 'RF2_den'}, inplace=True)
    for c in count_cols:
        out[c] = out[f'{c}_num'] / out[f'{c}_den'] * 100
    out['RF2'] = out['RF2_num'] / out['RF2_den'] * 100
    num_bases = {c[:-4] for c in out.columns if c.endswith('_num')}
    den_bases = {c[:-4] for c in out.columns if c.endswith('_den')}
    bases = sorted(num_bases | den_bases)
    rows = []
    for _, row in out.iterrows():
        for b in bases:
            rows.append({
                'indicador': b,
                'num':   row.get(f'{b}_num', pd.NA),
                'den':   row.get(f'{b}_den', pd.NA),
                'valor': row.get(b, pd.NA)  # la columna sin sufijo
            })
    out = pd.DataFrame(rows)
    out[['num', 'den']] = out[['num', 'den']].astype('Int64')
    out.loc[out['valor'].isna(), 'valor'] = out['num']

    # Ordenando
    orden_manual = ['codlocal']
    sub_gi = sorted([x for x in out['indicador'].unique() if 
                    re.match(r'^GI\d+_\d+', x)])
    gi_global = sorted([x for x in out['indicador'].unique() if 
                        re.match(r'^GI\d+$', x)])
    otros = ['ET1', 'ET2', 'OE1', 'OE2', 'OE4', 'RF1', 'RF2']
    orden_final = orden_manual + sub_gi + gi_global + otros
    out['indicador'] = pd.Categorical(out['indicador'], categories=orden_final, 
                                    ordered=True)
    out = out.sort_values('indicador').reset_index(drop=True)

    # Añadiendo año en las columnas
    out.rename(columns={'num': f'num_{anio}', 
                        'den': f'den_{anio}', 
                        'valor': f'valor_{anio}'}, 
            inplace=True)
    
    # Concatenar al dataframe final
    ind = pd.merge(ind, out, on=['indicador'], how='left') if not ind.empty \
        else out

ind.to_excel("data/procesadas/indicadores_PNIE.xlsx", index=False)

,indicador,num_PNIE,den_PNIE,valor_PNIE,num_2019,den_2019,valor_2019,num_2020,den_2020,valor_2020,...,valor_2022,num_2023,den_2023,valor_2023,num_2024,den_2024,valor_2024,num_2025,den_2025,valor_2025
0,codlocal,42331,<NA>,42331,54973,<NA>,54973,55058,<NA>,55058,...,55304,55358,<NA>,55358,55425,<NA>,55425,55602,<NA>,55602
1,GI1_1,492,27726,1.8,2725,30800,8.8,2076,30496,6.8,...,6.2,1749,37219,4.7,1360,34672,3.9,1509,34717,4.3
2,GI1_2,52,9811,0.5,887,10662,8.3,884,10637,8.3,...,10.2,842,8720,9.7,762,8258,9.2,710,8196,8.7
3,GI1_3,25,6410,0.4,592,6717,8.8,510,6613,7.7,...,9.0,449,5160,8.7,457,6088,7.5,387,6091,6.4
4,GI1_4,0,944,0.0,373,6344,5.9,384,8238,4.7,...,3.6,397,13543,2.9,288,12210,2.4,278,12204,2.3
5,GI1_5,113,12284,0.9,1933,13534,14.3,1503,13073,11.5,...,17.4,1381,8692,15.9,1604,7845,20.4,1422,7713,18.4
6,GI2_1,12341,40227,30.7,27189,53384,50.9,30272,54833,55.2,...,41.6,21559,55144,39.1,24284,55253,44.0,24164,55212,43.8
7,GI2_2,112,39922,0.3,2526,24652,10.2,2111,24557,8.6,...,15.1,3705,23505,15.8,4016,26703,15.0,3766,26748,14.1
8,GI3_1,106,39411,0.3,1506,15939,9.4,1909,24388,7.8,...,13.8,3209,22399,14.3,3565,25849,13.8,3285,25854,12.7
9,GI3_2,302,39352,0.8,4281,44571,9.6,3275,44339,7.4,...,8.4,7538,51189,14.7,5062,50113,10.1,7436,50127,14.8


## 04. Inversiones y gasto en la mejora de la infraestructura educativa

### 4.1 Procesamiendo de insumos

In [131]:
file = 'data/procesadas/banco_inversiones_criterios.csv'
bi = pd.read_csv(file, dtype={'CODIGO_INVERSION': 'string'})

file = 'data/procesadas/vinculaciones.csv'
vinc = pd.read_csv(file, dtype={'CUI': 'string', 'codlocal': 'string'})
vinc = vinc[~((vinc['CUI'] == '2040269') & (vinc['codlocal'] == '329667'))]
vinc = vinc[~((vinc['CUI'] == '2040272') & (vinc['codlocal'] == '329667'))]
vinc = vinc[~((vinc['CUI'] == '2040273') & (vinc['codlocal'] == '329667'))]
vinc = vinc[~(vinc['CUI'] == '2488226')]

C:\Users\Paolo\AppData\Local\Temp\ipykernel_9452\1365191188.py:2: DtypeWarning: Columns (136) have mixed types. Specify dtype option on import or set low_memory=False.
  bi = pd.read_csv(file, dtype={'CODIGO_INVERSION': 'string'})


In [ ]:
peip = pd.read_csv('data/procesadas/peip_intervenciones.csv',
                   dtype={'codlocal': str, 'CUI': str})

peip['GI5_1'] = 1
peip['fuente'] = 'PEIP-EB (jul 2025)'
unique(peip, ['codlocal', 'CUI', 'anio_culm'])

Number of unique values of codlocal CUI anio_culm is 45
Number of records is 45


: 

In [ ]:
ugeo = pd.read_csv('data/procesadas/pronied_ugeo_pre.csv',
                   dtype={'CUI': 'string', 'codlocal': 'string'})

ugeo['_val'] = 1
ugeo = (ugeo.pivot_table(index=['codlocal', 'CUI', 'anio_culm'],
                         columns='ind', values='_val', aggfunc='sum', 
                         fill_value=0).reset_index())
ugeo['fuente'] = 'PRONIED-UGEO'
unique(ugeo, ['codlocal', 'CUI', 'anio_culm'])

Number of unique values of codlocal CUI anio_culm is 168
Number of records is 168


: 

In [ ]:
ugrd_mbr = pd.read_csv('data/procesadas/pronied_ugrd_mbr.csv',
                       dtype={'FUR': 'string', 'codlocal': 'string'})

ugrd_mbr['_val'] = 1
ugrd_mbr = (ugrd_mbr.pivot_table(index=['codlocal', 'FUR', 'anio_culm'],
                                  columns='ind', values='_val', aggfunc='sum', 
                                  fill_value=0).reset_index())
ugrd_mbr.rename(columns={'FUR': 'CUI'}, inplace=True)
ugrd_mbr['fuente'] = 'PRONIED-UGRD-MBR'
unique(ugrd_mbr, ['codlocal', 'CUI', 'anio_culm'])

Number of unique values of codlocal CUI anio_culm is 55
Number of records is 55


: 

In [ ]:
ugrd_me = pd.read_csv('data/procesadas/pronied_ugrd_me.csv',
                      dtype={'FUR': 'string', 'codlocal': 'string'})

ugrd_me.rename(columns={'FUR': 'CUI'}, inplace=True)
ugrd_me['fuente'] = 'PRONIED-UGRD-ME'
unique(ugrd_me, ['codlocal', 'CUI', 'anio_culm'])

Number of unique values of codlocal CUI anio_culm is 15
Number of records is 15


: 

In [ ]:
file1 = 'data/Intervenciones/'
file2 = 'Anexo 3 - UGME.xlsx'
df_0 = pd.read_excel(file1 + file2, sheet_name='Sistemas modulares')

df = df_0.copy()

# Seleccionando información
df.columns = df.iloc[1, :]
df = df[2:].reset_index(drop=True)

# Corrigiendo código de local
codlocal(df, 'Código del local educativo', 'codlocal')
nocod = ['No tiene']
df = df[~df['codlocal'].isin(nocod)]

# Año de culminación
df['fec_culm'] = pd.to_datetime(
    df['Fecha de entrega de los módulos (o estimada)'], 
    errors='coerce')
df['anio_culm'] = df['fec_culm'].dt.year.astype('Int64', errors='ignore')
df = df.loc[df['anio_culm'] <= 2025, :]

# Indicadores
# No se consideran kits de pararrayos
df = df[df['Descripción del bien (tipologías)'] != 'KIT DE PARARRAYO']

# Acceso a agua y alcantarillado
df['acceso_agua'] = np.where(
    df['Descripción del bien (tipologías)'] == 'KIT DE RED DE AGUA', 1, 0)
df['GI2_2'] = np.where(
    df['Descripción del bien (tipologías)'] == 'KIT DE RED DE AGUA', 1, 0)
df['acceso_alcantarillado'] = np.where(
    df['Descripción del bien (tipologías)'] == 'KIT DE RED DE DESAGUE', 1, 0)
df['GI2_1'] = np.where(
    (df['acceso_agua'] == 1) & (df['acceso_alcantarillado'] == 1), 1, 0
)

# Módulos de SS.HH. (GI2.2)
sshh = ['KIT BAÑOS - A', 'MODULO PREFABRICADO DE SS.HH. TIPO COSTA',
        'MODULO PREFABRICADO DE SS.HH. TIPO HELADAS',
        'MODULO PREFABRICADO DE SS.HH. TIPO SELVA',
        'MODULO PREFABRICADO DE SS.HH. TIPO SIERRA']
df.loc[df['Descripción del bien (tipologías)'].isin(sshh), 'GI2_2'] = 1

# Módulos Plan Selva
df['GI1_4'] = np.where(
    df['¿La intervención implicó la instalación de módulos tipo Plan Selva?\n(SÍ / NO)'] == 'SI',
    1, 0)

# Instalación de aulas provisionales
df['aux'] = df[['acceso_agua', 'acceso_alcantarillado', 'GI2_1', 'GI2_2', 
                'GI1_4']].sum(axis=1)
df['GI1_1'] = np.where(df['aux'] == 0, 1, 0)
df.loc[df['GI1_4'] == 1, 'GI1_1'] = 1


# Colapsando a nivel de local educativo y año de culminación
df = df.groupby(['codlocal', 'anio_culm'], as_index=False).agg({
    'GI1_1': 'max',
    'GI1_4': 'max',
    'GI2_1': 'max',
    'GI2_2': 'max',
    'Monto contractual\n(S/)': 'sum'
})
df.rename(columns={'Monto contractual\n(S/)': 'monto_noinv'}, inplace=True)

ugme_sm = df.copy()
ugme_sm['fuente'] = 'PRONIED-UGME-SM'
unique(ugme_sm, ['codlocal', 'anio_culm'])

Number of unique values of codlocal anio_culm is 2,880
Number of records is 2,880


: 

In [ ]:
ugme_me = pd.read_csv('data/procesadas/pronied_ugme_me_prev.csv',
                      dtype={'cui_mob_equ': 'string', 'codlocal': 'string'})

# Eliminando intervenciones no escolarizadas
noes = ['000002', '000003', '3301315', '3950405', '0547458', '0592746',
        '0657111']
ugme_me = ugme_me[~ugme_me['codlocal'].isin(noes)]
# Indicador
ugme_me['ind'] = ''
ugme_me.loc[ugme_me['EQUIPAMIENTO'] > 0, 'ind'] = 'GI3_2'
ugme_me.loc[ugme_me['MOBILIARIO'] > 0, 'ind'] = 'GI3_2'

ugme_me.rename(columns={'cui_mob_equ': 'CUI'}, inplace=True)
ugme_me['_val'] = 1
ugme_me = (ugme_me.pivot_table(index=['codlocal', 'CUI', 'anio_culm'],
                               columns='ind', values='_val', aggfunc='sum', 
                               fill_value=0).reset_index())
ugme_me['fuente'] = 'PRONIED-UGME-ME'
unique(ugme_me, ['codlocal', 'CUI', 'anio_culm'])

Number of unique values of codlocal CUI anio_culm is 1,876
Number of records is 1,876


: 

In [ ]:
file1 = 'data/Intervenciones/'
file2 = 'Anexo 4_UGM.xlsx'
df_0 = pd.read_excel(file1 + file2, sheet_name='Accesibilidad (2017-2025)')

df = df_0.copy()

# Seleccionando información
df.columns = df.iloc[1, :]
df = df[2:-7].reset_index(drop=True)
codlocal(df, 'Código del local educativo', 'codlocal')
df.drop(columns=['N°'], inplace=True)
df.drop_duplicates(inplace=True)

# Indicador de accesibilidad
df['GI4_3'] = np.where((df[df.columns[9]] == 'SI') | 
                       (df[df.columns[10]] == 'SI'), 1, 0)
df = df[df['GI4_3'] == 1]

# Colapsando a nivel de codlocal
df.rename(columns={'Año': 'anio_culm'}, inplace=True)

df = df.groupby('codlocal', as_index=False).agg({
    'anio_culm': 'max',
    'GI4_3': 'max',
    'Monto incurrido en actividades de accesibilidad\n(S/)\n(*)': 'sum'
})
df.rename(columns={'Monto incurrido en actividades de accesibilidad\n(S/)\n(*)': 'monto_noinv'}, inplace=True)

ugm_acc = df.copy()
ugm_acc['fuente'] = 'PRONIED-UGM-ACCESIBILIDAD'
unique(ugm_acc, ['codlocal', 'anio_culm'])

Number of unique values of codlocal anio_culm is 671
Number of records is 671


: 

In [ ]:
anios = ['2019', '2020', '2021', '2022', '2023', '2024', '2025']

mps = pd.DataFrame()
for anio in anios:
    df = pd.read_csv(f"data/procesadas/indicadores_{anio}.csv", 
                     dtype={'codlocal': str})
    df = df[['codlocal',  'GI3_4']]
    df['anio_culm'] = anio
    df = df.loc[df['GI3_4'] == 1]
    mps = pd.concat([mps, df], ignore_index=True)

ugm_mp = pd.read_csv('data/procesadas/pronied_ugm_mto_preventivo.csv',
                     dtype={'anio_culm': 'string', 'codlocal': 'string'})
ugm_mp.rename(columns={'monto_mp': 'monto_noinv'}, inplace=True)
ugm_mp = pd.merge(ugm_mp, mps, on=['codlocal', 'anio_culm'], how='left')
ugm_mp['GI3_4'].fillna(0, inplace=True)

ugm_mp['fuente'] = 'PRONIED-UGM-PREVENTIVO'
unique(ugm_mp, ['codlocal', 'anio_culm'])

Number of unique values of codlocal anio_culm is 422,718
Number of records is 422,718


C:\Users\Paolo\AppData\Local\Temp\ipykernel_9452\2954627010.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  ugm_mp['GI3_4'].fillna(0, inplace=True)


: 

In [ ]:
ugsc_asitec = pd.read_csv('data/procesadas/pronied_ugsc_asitec.csv',
                      dtype={'CUI': 'string', 'codlocal': 'string'})

ugsc_asitec['_val'] = 1
ugsc_asitec = (ugsc_asitec.pivot_table(index=['codlocal', 'CUI', 'anio_culm'],
                                       columns='ind', values='_val', 
                                       aggfunc='sum', fill_value=0). \
                                        reset_index())

ugsc_asitec['fuente'] = 'PRONIED-UGSC-ASITEC/SIAT'
unique(ugsc_asitec, ['codlocal', 'anio_culm'])

Number of unique values of codlocal anio_culm is 127
Number of records is 127


: 

In [ ]:
ugsc_sym = pd.read_csv('data/procesadas/pronied_ugsc_sym.csv',
                      dtype={'CUI': 'string', 'codlocal': 'string'})

ugsc_sym['_val'] = 1
ugsc_sym = (ugsc_sym.pivot_table(index=['codlocal', 'CUI', 'anio_culm'],
                                 columns='ind', values='_val', aggfunc='sum', 
                                 fill_value=0).reset_index())

ugsc_sym['fuente'] = 'PRONIED-UGSC-SyM'
unique(ugsc_sym, ['codlocal', 'anio_culm'])

Number of unique values of codlocal anio_culm is 126
Number of records is 126


: 

In [ ]:
anin_mto = pd.read_csv('data/procesadas/anin_mantenimiento.csv',
                      dtype={'CUI': 'string', 'codlocal': 'string'})

anin_mto['_val'] = 1
anin_mto = (anin_mto.pivot_table(index=['codlocal', 'CUI', 'anio_culm'],
                                 columns='ind', values='_val', aggfunc='sum', 
                                 fill_value=0).reset_index())
anin_mto['monto_noinv'] = np.nan
anin_mto['fuente'] = 'ANIN'
unique(anin_mto, ['codlocal', 'CUI', 'anio_culm'])

Number of unique values of codlocal CUI anio_culm is 28
Number of records is 28


: 

In [ ]:
pircc = pd.read_csv('data/procesadas/pircc_arcc_anin.csv',
                    dtype={'CUI': 'string', 'codlocal': 'string'})
pircc['anio_culm'] = pircc['anio_culm'].astype(int).astype(str)

pircc['_val'] = 1
pircc = (pircc.pivot_table(index=['codlocal', 'CUI', 'anio_culm'],
                           columns='ind', values='_val', aggfunc='sum', 
                           fill_value=0).reset_index())
pircc['fuente'] = 'ARCC / ANIN'
unique(pircc, ['codlocal', 'anio_culm'])

Number of unique values of codlocal anio_culm is 830
Number of records is 830


: 

In [ ]:
file1 = 'data/Intervenciones/DIGESE.xlsx'
df_0 = pd.read_excel(file1)

df = df_0.copy()

# Corrección COAR Cajamarca
codlocal(df, 'Código local', 'codlocal')
df.loc[(df['codlocal'] == '394605') & 
       (df['Responsable'] == 'GORE CAJAMARCA'), 'codlocal'] = '097733'

# Supuesto 1: Solo el mantenimiento correctivo y preventivo es el adecuado
# Supuesto 2: El monto de mto. preventivo se divide entre años de periodo
mtos = ['PREVENTIVO', 'CORRECTIVO']
df = df.loc[df['Tipo de mantenimiento'].isin(mtos)]

# Año de culminación
df['anio_culm'] = 2025 # Supuesto: Intervenciones de MTO-C son recientes
df.loc[df['Tipo de mantenimiento'] == 'PREVENTIVO', 'anio_culm'] = df['Año']

# Elaborando los indicadores del GI3.3
df['ind'] = np.where(df['Tipo de mantenimiento'] == 'PREVENTIVO',
                     'GI3_4', 'GI3_3')
df.rename(columns={'Costo': 'monto_noinv'}, inplace=True)

df['_val'] = 1
df = (df.pivot_table(index=['codlocal', 'anio_culm', 'monto_noinv'],
                         columns='ind', values='_val', aggfunc='sum', 
                     fill_value=0).reset_index())
digese = df.copy()
digese['fuente'] = 'DIGESE'
unique(digese, ['codlocal', 'anio_culm'])

Number of unique values of codlocal anio_culm is 19
Number of records is 19


: 

In [ ]:
file = 'data/procesadas/vinculaciones.csv'
vinc = pd.read_csv(file, dtype={'CUI': 'string', 'codlocal': 'string'})

file1 = 'data/Intervenciones/FONCODES.xlsx'
df_0 = pd.read_excel(file1)

df = df_0.copy()

# Seleccionando información
df.columns = df.iloc[1, :]
df = df.iloc[2:-1, 1:].reset_index(drop=True)

# Solo se consideran proyectos con CUI identificados
df = df[~(df['CUI'] == 'No tiene')]

# Año de culminación
df['fec_culm'] = pd.to_datetime(df['FECHA TERMINO'], errors='coerce')
df['anio_culm'] = df['fec_culm'].dt.year.astype('Int64', errors='ignore')
df = df.loc[df['anio_culm'] <= 2025, :]

# Vinculaciones con los locales educativos
df = df[['CUI', 'anio_culm']]
df = pd.merge(df, vinc[['CUI', 'codlocal']],
              on='CUI', how='left')

# Indicadores: Ampliación de infraestructura
df['GI4_4'] = 1

foncodes = df.copy()
foncodes['fuente'] = 'FONCODES'
unique(foncodes, ['codlocal', 'anio_culm'])

Number of unique values of codlocal anio_culm is 5
Number of records is 5


: 

In [ ]:
df2 = pd.read_csv('data/procesadas/otros_cui_gn_final.csv',
                 dtype={'CUI': 'string', 'codlocal': 'string'})

df2 = pd.merge(df2, bi[['CODIGO_INVERSION', 'FUNCION', 'PROGRAMA',
                        'SUB_PROGRAMA']],
               left_on='CUI', right_on='CODIGO_INVERSION', how='left')

# Eliminando intervenciones con FUNCION en ciencia y tecnologia
df2 = df2.loc[~(df2['FUNCION'] == 'CIENCIA Y TECNOLOGÍA')]

# Eliminando intervenciones con FUNCION en cultura y deporte
df2 = df2.loc[~(df2['FUNCION'] == 'CULTURA Y DEPORTE')]
df2 = df2.loc[~(df2['FUNCION'] == 'DEPORTES')]

# Eliminando intervenciones con FUNCION en educacion y PROGRAMA en asistencia
# educativa
df2 = df2.loc[~((df2['FUNCION'] == 'EDUCACIÓN') & 
                 (df2['PROGRAMA'] == 'ASISTENCIA EDUCATIVA'))]

# Eliminando intervenciones con FUNCION en educacion y cultura y PROGRAMA en
# asistencia a educandos, capacitacion o educacion fisica y deportes
df2 = df2.loc[~((df2['FUNCION'] == 'EDUCACION Y CULTURA') & 
                 (df2['PROGRAMA'] == 'ASISTENCIA A EDUCANDOS'))]
df2 = df2.loc[~((df2['FUNCION'] == 'EDUCACION Y CULTURA') & 
                 (df2['PROGRAMA'] == 
                  'CAPACITACION Y PERFECCIONAMIENTO'))]
df2 = df2.loc[~((df2['FUNCION'] == 'EDUCACION Y CULTURA') & 
                 (df2['PROGRAMA'] == 'EDUCACION FISICA Y DEPORTES'))]

# Eliminando intervencniones en orden publico y seguridad
df2 = df2.loc[~(df2['FUNCION'] == 'ORDEN PÚBLICO Y SEGURIDAD')]

# Eliminando intervenciones en planeamiento, gestion y reserva de contingencia
df2 = df2.loc[~(df2['FUNCION'] == 
                 'PLANEAMIENTO, GESTIÓN Y RESERVA DE CONTINGENCIA')]

# Eliminando intervenciones en salud
df2 = df2.loc[~(df2['FUNCION'] == 'SALUD')]

# Solo se considera intervenciones identificadas en el PNIE
df2['suma'] = df2[['GI1_1', 'GI1_2', 'GI1_3', 'GI1_4', 'GI1_5', 
                   'GI2_1', 'GI2_2', 
                   'GI3_1', 'GI3_2', 
                   'GI4_1', 'GI4_2', 'GI4_3', 'GI4_4', 
                   'GI5_1', 'GI5_2']].sum(axis=1)
df2 = df2[df2['suma'] > 0].drop(columns=['suma'])

# Transformando a nivel de codlocal y año de culminación de la intervención
gn = pd.merge(df2, vinc[['CUI', 'codlocal']],
                 on='CUI', how='left', indicator='CUI_codlocal')
gn = gn[gn['CUI_codlocal'] == 'both'].drop(columns=['CUI_codlocal'])
gn = (
    gn.groupby(['codlocal', 'anio_culm', 'CUI'], as_index=False)
        .agg({
            'GI1_1': 'max', 'GI1_2': 'max', 'GI1_3': 'max', 'GI1_4': 'max', 
            'GI1_5': 'max', 
            'GI2_1': 'max', 'GI2_2': 'max', 
            'GI3_1': 'max', 'GI3_2': 'max', 
            'GI4_1': 'max', 'GI4_2': 'max', 'GI4_3': 'max', 'GI4_4': 'max', 
            'GI5_1': 'max', 'GI5_2': 'max',
            'fuente': lambda x: " / ".join(sorted(x.astype(str).unique()))
        })
)

unique(gn, ['codlocal', 'CUI', 'anio_culm'])

Number of unique values of codlocal CUI anio_culm is 4,848
Number of records is 4,848


: 

In [ ]:
subnac = pd.read_csv('data/procesadas/bi_cui_gr_gl_final.csv',
                 dtype={'CUI': 'string', 'codlocal': 'string'})

subnac = pd.merge(subnac, bi[['CODIGO_INVERSION', 'FUNCION', 'PROGRAMA',
                             'SUB_PROGRAMA']],
                  left_on='CUI', right_on='CODIGO_INVERSION', how='left')

# Eliminando intervenciones con FUNCION en ciencia y tecnologia
subnac = subnac.loc[~(subnac['FUNCION'] == 'CIENCIA Y TECNOLOGÍA')]

# Eliminando intervenciones con FUNCION en cultura y deporte
subnac = subnac.loc[~(subnac['FUNCION'] == 'CULTURA Y DEPORTE')]
subnac = subnac.loc[~(subnac['FUNCION'] == 'DEPORTES')]

# Eliminando intervenciones con FUNCION en educacion y PROGRAMA en asistencia
# educativa
subnac = subnac.loc[~((subnac['FUNCION'] == 'EDUCACIÓN') & 
                      (subnac['PROGRAMA'] == 'ASISTENCIA EDUCATIVA'))]

# Eliminando intervenciones con FUNCION en educacion y cultura y PROGRAMA en
# asistencia a educandos, capacitacion o educacion fisica y deportes
subnac = subnac.loc[~((subnac['FUNCION'] == 'EDUCACION Y CULTURA') & 
                      (subnac['PROGRAMA'] == 'ASISTENCIA A EDUCANDOS'))]
subnac = subnac.loc[~((subnac['FUNCION'] == 'EDUCACION Y CULTURA') & 
                      (subnac['PROGRAMA'] == 
                       'CAPACITACION Y PERFECCIONAMIENTO'))]
subnac = subnac.loc[~((subnac['FUNCION'] == 'EDUCACION Y CULTURA') & 
                      (subnac['PROGRAMA'] == 'EDUCACION FISICA Y DEPORTES'))]

# Eliminando intervencniones en orden publico y seguridad
subnac = subnac.loc[~(subnac['FUNCION'] == 'ORDEN PÚBLICO Y SEGURIDAD')]

# Eliminando intervenciones en planeamiento, gestion y reserva de contingencia
subnac = subnac.loc[~(subnac['FUNCION'] == 
                      'PLANEAMIENTO, GESTIÓN Y RESERVA DE CONTINGENCIA')]

# Eliminando intervenciones en salud
subnac = subnac.loc[~(subnac['FUNCION'] == 'SALUD')]

# Solo se considera intervenciones identificadas en el PNIE
subnac['suma'] = subnac[['GI1_1', 'GI1_2', 'GI1_3', 'GI1_4', 'GI1_5', 
                         'GI2_1', 'GI2_2', 
                         'GI3_1', 'GI3_2', 
                         'GI4_1', 'GI4_2', 'GI4_3', 'GI4_4', 
                         'GI5_1', 'GI5_2']].sum(axis=1)
subnac = subnac[subnac['suma'] > 0].drop(columns=['suma'])

# Transformando a nivel de codlocal y año de culminación de la intervención
gr_gl = pd.merge(subnac, vinc[['CUI', 'codlocal']],
                 on='CUI', how='left', indicator='CUI_codlocal')
gr_gl = gr_gl[gr_gl['CUI_codlocal'] == 'both'].drop(columns=['CUI_codlocal'])

gr_gl = (
    gr_gl.groupby(['codlocal', 'anio_culm', 'CUI'], as_index=False)
        .agg({
            'GI1_1': 'max', 'GI1_2': 'max', 'GI1_3': 'max', 'GI1_4': 'max', 
            'GI1_5': 'max', 
            'GI2_1': 'max', 'GI2_2': 'max', 
            'GI3_1': 'max', 'GI3_2': 'max', 
            'GI4_1': 'max', 'GI4_2': 'max', 'GI4_3': 'max', 'GI4_4': 'max', 
            'GI5_1': 'max', 'GI5_2': 'max',
            'fuente': lambda x: " / ".join(sorted(x.astype(str).unique()))
        })
)

unique(gr_gl, ['codlocal', 'CUI', 'anio_culm'])

Number of unique values of codlocal CUI anio_culm is 21,046
Number of records is 21,046


: 

### 4.2 Consolidación de insumos

In [132]:
df = pd.DataFrame()

inputs = [peip, ugeo, ugrd_mbr, ugrd_me, ugme_sm, ugme_me, ugm_acc, ugm_mp, 
          ugsc_asitec, ugsc_sym, anin_mto, pircc, digese, foncodes, gn, gr_gl]

for df_input in inputs:
    df = pd.concat([df, df_input], ignore_index=True)

df = df[['codlocal', 'CUI', 'anio_culm',
         'GI1_1', 'GI1_2', 'GI1_3', 'GI1_4', 'GI1_5', 
         'GI2_1', 'GI2_2',
         'GI3_1', 'GI3_2', 'GI3_3', 'GI3_4', 
         'GI4_1', 'GI4_2', 'GI4_3', 'GI4_4',
         'GI5_1', 'GI5_2', 'monto_noinv', 'fuente']] 
df.loc[df['CUI'].isna(), 'CUI'] = ''
cols = ['GI1_1', 'GI1_2', 'GI1_3', 'GI1_4', 'GI1_5', 
        'GI2_1', 'GI2_2',
        'GI3_1', 'GI3_2', 'GI3_3', 'GI3_4',
        'GI4_1', 'GI4_2', 'GI4_3', 'GI4_4',
        'GI5_1', 'GI5_2']
df[cols] = df[cols].fillna(0)

# Considerando solo las intervenciones desde 2017
df['anio_culm'] = df['anio_culm'].astype(int)
# df = df[df['anio_culm'] >= 2017]

# Colapsando a nivel de codlocal-cui-anio culminacion
df = df.groupby(['codlocal', 'CUI', 'anio_culm'], as_index=False).agg({
    'GI1_1': 'sum', 'GI1_2': 'sum', 'GI1_3': 'sum', 'GI1_4': 'sum', 
    'GI1_5': 'sum',
    'GI2_1': 'sum', 'GI2_2': 'sum',
    'GI3_1': 'sum', 'GI3_2': 'sum', 'GI3_3': 'sum', 'GI3_4': 'sum',
    'GI4_1': 'sum', 'GI4_2': 'sum', 'GI4_3': 'sum', 'GI4_4': 'sum',
    'GI5_1': 'sum', 'GI5_2': 'sum', 
    'monto_noinv': 'sum', 'fuente': lambda x: ' / '.join(x)
})

# Agregando información del banco de inversiones
df = pd.merge(df,
              bi[['CODIGO_INVERSION', 'NOMBRE_INVERSION', 'DES_TIPO_FORMATO', 
                  'TIPO_IOARR', 'ESTADO', 'SITUACION', 
                  'NIVEL', 'UEP_ULTIMA', 'UEP_PRINCIPAL', 'UF', 'UEI', 
                  'COSTO_ACTUALIZADO_BI', 'COSTO_INV_TOTAL_BI']],
              left_on='CUI', right_on='CODIGO_INVERSION', how='left')
df.drop(columns=['CODIGO_INVERSION'], inplace=True)

# Completando la información de NIVEL
gns = ['PRONIED-UGM-PREVENTIVO', 'PRONIED-UGM-ACCESIBILIDAD', 'PRONIED-UGME-SM', 
       'PRONIED-UGME-ME', 'PRONIED-UGEO', 'DIGESE',
       'PRONIED-UGME-SM / PRONIED-UGM-ACCESIBILIDAD',
       'PRONIED-UGM-ACCESIBILIDAD / PRONIED-UGM-PREVENTIVO',
       'PRONIED-UGME-SM / PRONIED-UGM-PREVENTIVO',
       'PRONIED-UGM-PREVENTIVO / DIGESE',
       'PRONIED-UGME-SM / PRONIED-UGM-ACCESIBILIDAD / PRONIED-UGM-PREVENTIVO']
df.loc[(df['NIVEL'].isna()) & (df['fuente'].isin(gns)), 'NIVEL'] = 'GN'

# Obteniendo información sobre LLEE nuevos y existente y correcciones
exis = pd.read_csv('data/procesadas/llee_existentes.csv', 
                   dtype={'codlocal': str})
df = pd.merge(df, exis, on='codlocal', how='left', indicator='_existente')


df.loc[(df['_existente'] == 'both') & (df['GI5_2'] == 1), 'GI5_2'] = 0
ind_exis = ['GI1_1', 'GI1_2', 'GI1_3', 'GI1_4', 'GI1_5', 'GI2_1', 'GI2_2',
            'GI3_1', 'GI3_2', 'GI3_3', 'GI3_4', 'GI4_1', 'GI4_2', 'GI4_3', 
            'GI4_4', 'GI5_1']
for ind in ind_exis:
    df.loc[(df['_existente'] == 'left_only') & (df[ind] == 1), ind] = 0
df.drop(columns=['_existente'], inplace=True)

# Correcciones en valores de indicadores: 0 o 1 excepto GI3_4
indis = ['GI1_1', 'GI1_2', 'GI1_3', 'GI1_4', 'GI1_5', 
         'GI2_1', 'GI2_2',
         'GI3_1', 'GI3_2', 'GI3_3', 'GI3_4',
         'GI4_1', 'GI4_2', 'GI4_3', 'GI4_4',
         'GI5_1', 'GI5_2']
for ind in indis:
    df.loc[(df[ind] > 1) & (df[ind].notna()), ind] = 1

anios = ['PNIE', '2018', '2019', '2020', '2021', '2022', '2023', '2024', 
         '2025']

for anio in anios:
    if anio == 'PNIE':
        lb_2017 = pd.read_stata(f'data/LE_BasePr_{anio}.dta')
        codlocal(lb_2017, 'id_local', 'codlocal')
        lb_2017 = lb_2017['codlocal']

    elif anio == '2018':
        lb_2018 = pd.read_stata(f'data/LE_BasePr_2019.dta')
        codlocal(lb_2018, 'cod_local', 'codlocal')
        lb_2018 = lb_2018['codlocal']

    else:
        globals()[f"lb_{anio}"] = pd.read_stata(f'data/LE_BasePr_{anio}.dta')
        codlocal(globals()[f"lb_{anio}"], 'cod_local', 'codlocal')
        globals()[f"lb_{anio}"] = globals()[f"lb_{anio}"]['codlocal']

anios = [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
df2 = pd.DataFrame()
for anio in anios:
    if anio == 2017:
        aux = df[df['anio_culm'] <= anio]
        aux = pd.merge(aux, globals()[f"lb_{anio}"], on='codlocal', how='left', 
                    indicator=f'_{anio}')
        aux['universo_cabrpr'] = np.where(aux[f'_{anio}'] == 'both', 1, 0)
        aux.drop(columns=f'_{anio}', inplace=True)
        df2 = pd.concat([df2, aux], ignore_index=True)

    else:
        aux = df[df['anio_culm'] == anio]
        aux = pd.merge(aux, globals()[f"lb_{anio}"], on='codlocal', how='left', 
                    indicator=f'_{anio}')
        aux['universo_cabrpr'] = np.where(aux[f'_{anio}'] == 'both', 1, 0)
        aux.drop(columns=f'_{anio}', inplace=True)
        df2 = pd.concat([df2, aux], ignore_index=True)

df = df2.copy()

# Completando CUI con intervenciones que no son inversiones
df['inv'] = np.where(df['CUI'] != '', 1, 0)
df.loc[df['CUI'] == 'Presupuesto por actividades', 'inv'] = 0
df.loc[(df['CUI'] == '') & (df['fuente'] == 'PRONIED-UGM-PREVENTIVO'),
       'CUI'] = 'Mantenimiento preventivo (UGM-PRONIED)'
df.loc[(df['CUI'] == '') & 
       (df['fuente'] == 'PRONIED-UGME-SM / PRONIED-UGM-PREVENTIVO'),
       'CUI'] = 'Mantenimiento preventivo (UGM-PRONIED) y sistemas modulares (UGME-PRONIED)'
df.loc[(df['CUI'] == '') & 
       (df['fuente'] == 'PRONIED-UGM-ACCESIBILIDAD / PRONIED-UGM-PREVENTIVO'),
       'CUI'] = 'Mantenimiento preventivo y accesibilidad (UGM-PRONIED)'
df.loc[(df['CUI'] == '') & 
       (df['fuente'] == 'PRONIED-UGME-SM'),
       'CUI'] = 'Sistemas modulares (UGME-PRONIED)'
df.loc[(df['CUI'] == '') & 
       (df['fuente'] == 'PRONIED-UGM-PREVENTIVO / DIGESE'),
       'CUI'] = 'Mantenimiento preventivo (UGM-PRONIED) y mantenimiento preventivo y/o correctivo (DIGESE)'
df.loc[(df['CUI'] == '') & 
       (df['fuente'] == 'PRONIED-UGM-ACCESIBILIDAD'),
       'CUI'] = 'Accesibilidad (UGM-PRONIED)'
df.loc[(df['CUI'] == '') & 
       (df['fuente'] == 'DIGESE'),
       'CUI'] = 'Mantenimiento preventivo y/o correctivo (DIGESE)'
df.loc[(df['CUI'] == '') & 
       (df['fuente'] == 'PRONIED-UGME-SM / PRONIED-UGM-ACCESIBILIDAD'),
       'CUI'] = 'Sistemas modulares (UGME-PRONIED) y accesibilidad (UGM-PRONIED)'
df.loc[(df['CUI'] == '') & 
       (df['fuente'] == 'PRONIED-UGME-SM / PRONIED-UGM-ACCESIBILIDAD / PRONIED-UGM-PREVENTIVO'),
       'CUI'] = 'Sistemas modulares (UGME-PRONIED), accesibilidad y mantenimiento preventivo (UGM-PRONIED)'

# Ordenando el dataframe
df.sort_values(by=['codlocal', 'CUI', 'anio_culm'], inplace=True)
df = df[['codlocal', 'CUI', 'anio_culm',
         'NOMBRE_INVERSION', 'DES_TIPO_FORMATO',
         'TIPO_IOARR', 'ESTADO', 'SITUACION', 'NIVEL', 'UEP_ULTIMA',
         'UEP_PRINCIPAL', 'UF', 'UEI', 
         'COSTO_ACTUALIZADO_BI', 'COSTO_INV_TOTAL_BI', 'monto_noinv', 
         'GI1_1', 'GI1_2', 'GI1_3', 'GI1_4', 'GI1_5', 
         'GI2_1', 'GI2_2', 
         'GI3_1', 'GI3_2', 'GI3_3', 'GI3_4', 
         'GI4_1', 'GI4_2', 'GI4_3', 'GI4_4', 
         'GI5_1', 'GI5_2',
         'fuente', 'universo_cabrpr', 'inv']].reset_index(drop=True)

unique(df, ['codlocal', 'CUI', 'anio_culm'])
unique(df, ['CUI'])
df.to_excel('data/procesadas/LE_CUI_anio_intervenciones_total_v0.xlsx', 
            index=False, sheet_name='Data')

C:\Users\Paolo\AppData\Local\Temp\ipykernel_9452\3871407433.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[newvar] = df[oldvar].astype(str).str.zfill(6)
C:\Users\Paolo\AppData\Local\Temp\ipykernel_9452\3871407433.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[newvar] = df[oldvar].astype(str).str.zfill(6)
C:\Users\Paolo\AppData\Local\Temp\ipykernel_9452\3871407433.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance. 

Number of unique values of codlocal CUI anio_culm is 452,119
Number of records is 452,119
Number of unique values of CUI is 11,058
Number of records is 452,119


In [133]:
df2 = df.copy()

# Colapsando a nivel de CUI y año de culminación
df2 = df2[df2['inv'] == 1]
df2.drop(columns=['codlocal', 'universo_cabrpr', 'inv', 'monto_noinv'], 
         inplace=True)
df2.drop_duplicates(inplace=True)

aux1 = df2[['CUI', 'anio_culm', 'NOMBRE_INVERSION', 'DES_TIPO_FORMATO',
       'TIPO_IOARR', 'ESTADO', 'SITUACION', 'NIVEL', 'UEP_ULTIMA',
       'UEP_PRINCIPAL', 'UF', 'UEI', 'COSTO_ACTUALIZADO_BI',
       'COSTO_INV_TOTAL_BI', 'fuente']]
aux2 = df2[['CUI', 'anio_culm',
            'GI1_1', 'GI1_2', 'GI1_3', 'GI1_4',
       'GI1_5', 'GI2_1', 'GI2_2', 'GI3_1', 'GI3_2', 'GI3_3', 'GI3_4', 'GI4_1',
       'GI4_2', 'GI4_3', 'GI4_4', 'GI5_1', 'GI5_2']]
aux2 = aux2.groupby(['CUI', 'anio_culm'], as_index=False).agg({
    'GI1_1': 'max', 'GI1_2': 'max', 'GI1_3': 'max', 'GI1_4': 'max',
    'GI1_5': 'max', 'GI2_1': 'max', 'GI2_2': 'max', 'GI3_1': 'max',
    'GI3_2': 'max', 'GI3_3': 'max', 'GI3_4': 'max', 'GI4_1': 'max',
    'GI4_2': 'max', 'GI4_3': 'max', 'GI4_4': 'max', 'GI5_1': 'max',
    'GI5_2': 'max'
})
df2 = pd.merge(aux1, aux2, on=['CUI', 'anio_culm'], how='left', indicator=True)
df2.drop_duplicates(inplace=True)
df2.drop(columns=['NOMBRE_INVERSION', 'DES_TIPO_FORMATO',
       'TIPO_IOARR', 'ESTADO', 'SITUACION', 'NIVEL', 'UEP_ULTIMA',
       'UEP_PRINCIPAL', 'UF', 'UEI', 'COSTO_ACTUALIZADO_BI',
       'COSTO_INV_TOTAL_BI', '_merge'], 
         inplace=True)
df2 = df2.groupby(['CUI'], as_index=False).agg({
    'anio_culm': 'max',
    'GI1_1': 'max', 'GI1_2': 'max', 'GI1_3': 'max', 'GI1_4': 'max',
    'GI1_5': 'max', 'GI2_1': 'max', 'GI2_2': 'max', 'GI3_1': 'max',
    'GI3_2': 'max', 'GI3_3': 'max', 'GI3_4': 'max', 'GI4_1': 'max',
    'GI4_2': 'max', 'GI4_3': 'max', 'GI4_4': 'max', 'GI5_1': 'max',
    'GI5_2': 'max',
    'fuente': lambda x: " / ".join(sorted(x.astype(str).unique()))
})
df2 = pd.merge(df2,
              bi[['CODIGO_INVERSION', 'NOMBRE_INVERSION', 'DES_TIPO_FORMATO', 
                  'TIPO_IOARR', 'ESTADO', 'SITUACION', 
                  'NIVEL', 'UEP_ULTIMA', 'UEP_PRINCIPAL', 'UF', 'UEI', 
                  'COSTO_ACTUALIZADO_BI', 'COSTO_INV_TOTAL_BI']],
              left_on='CUI', right_on='CODIGO_INVERSION', how='left')
df2.drop(columns=['CODIGO_INVERSION'], inplace=True)

# Ordenando el dataframe
df2.sort_values(by=['CUI', 'anio_culm'], inplace=True)
df2 = df2[['CUI', 'anio_culm',
         'NOMBRE_INVERSION', 'DES_TIPO_FORMATO',
         'TIPO_IOARR', 'ESTADO', 'SITUACION', 'NIVEL', 'UEP_ULTIMA',
         'UEP_PRINCIPAL', 'UF', 'UEI', 
         'COSTO_ACTUALIZADO_BI', 'COSTO_INV_TOTAL_BI', 
         'GI1_1', 'GI1_2', 'GI1_3', 'GI1_4', 'GI1_5', 
         'GI2_1', 'GI2_2', 
         'GI3_1', 'GI3_2', 'GI3_3', 'GI3_4', 
         'GI4_1', 'GI4_2', 'GI4_3', 'GI4_4', 
         'GI5_1', 'GI5_2',
         'fuente']].reset_index(drop=True)

unique(df2, ['CUI'])
df2.to_excel('data/procesadas/CUI_anio_intervenciones_total_v0.xlsx', 
            index=False, sheet_name='Data')

Number of unique values of CUI is 11,048
Number of records is 11,048


In [138]:
df.loc[df['CUI'] == '2155697']

,codlocal,CUI,anio_culm,NOMBRE_INVERSION,DES_TIPO_FORMATO,TIPO_IOARR,ESTADO,SITUACION,NIVEL,UEP_ULTIMA,...,GI3_4,GI4_1,GI4_2,GI4_3,GI4_4,GI5_1,GI5_2,fuente,universo_cabrpr,inv
209098,348996,2155697,2016,"SUSTITUCIÓN, REFORZAMIENTO Y MEJORAMIENTO DE L...",PROYECTO DE INVERSION,NaN,ACTIVO,VIABLE,GN,M.E.-PROGRAMA NACIONAL DE INFRAESTRUCTURA EDU...,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,PRONIED-UGEO,1,1
209099,348996,2155697,2021,"SUSTITUCIÓN, REFORZAMIENTO Y MEJORAMIENTO DE L...",PROYECTO DE INVERSION,NaN,ACTIVO,VIABLE,GN,M.E.-PROGRAMA NACIONAL DE INFRAESTRUCTURA EDU...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,PRONIED-UGME-ME,1,1
209100,348996,2155697,2024,"SUSTITUCIÓN, REFORZAMIENTO Y MEJORAMIENTO DE L...",PROYECTO DE INVERSION,NaN,ACTIVO,VIABLE,GN,M.E.-PROGRAMA NACIONAL DE INFRAESTRUCTURA EDU...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,PRONIED-UGME-ME,1,1


In [116]:
df2.columns

Index(['CUI', 'anio_culm', 'GI1_1', 'GI1_2', 'GI1_3', 'GI1_4', 'GI1_5',
       'GI2_1', 'GI2_2', 'GI3_1', 'GI3_2', 'GI3_3', 'GI3_4', 'GI4_1', 'GI4_2',
       'GI4_3', 'GI4_4', 'GI5_1', 'GI5_2', 'fuente', 'NOMBRE_INVERSION',
       'DES_TIPO_FORMATO', 'TIPO_IOARR', 'ESTADO', 'SITUACION', 'NIVEL',
       'UEP_ULTIMA', 'UEP_PRINCIPAL', 'UF', 'UEI', 'COSTO_ACTUALIZADO_BI',
       'COSTO_INV_TOTAL_BI'],
      dtype='object')

In [ ]:
df[df['codlocal', 'CUI', 'anio_culm'].duplicated(keep=False)].sort_values(by='codlocal')

: 